In [1]:

# Stage 5 - Load feature-engineered data

from google.colab import drive
import pandas as pd

drive.mount("/content/drive")

folder = "/content/drive/MyDrive/LendingClub_Project/stage4_data"

train_df = pd.read_parquet(
    folder + "/lending_club_train_stage4.parquet"
)

test_df = pd.read_parquet(
    folder + "/lending_club_test_stage4.parquet"
)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)
print("Same columns:", train_df.columns.equals(test_df.columns))
print("Missing values:", train_df.isna().sum().sum(),
      test_df.isna().sum().sum())


Mounted at /content/drive
Training shape: (1076280, 27)
Testing shape: (269070, 27)
Same columns: True
Missing values: 0 0


In [2]:

# Stage 5 - Identify numerical and categorical features

X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Numerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))

print("\nCategorical columns:")
print(categorical_cols)

print("\nTarget distribution:")
print(y_train.value_counts())


Numerical features: 20
Categorical features: 6

Categorical columns:
['sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'addr_state']

Target distribution:
target
0    861401
1    214879
Name: count, dtype: int64


In [3]:

# Stage 5 - Check categorical feature sizes

for col in categorical_cols:
    print(f"{col}: {X_train[col].nunique()} unique categories")

print("\nTotal training records:", len(X_train))


sub_grade: 35 unique categories
emp_length: 12 unique categories
home_ownership: 6 unique categories
verification_status: 3 unique categories
purpose: 14 unique categories
addr_state: 51 unique categories

Total training records: 1076280


In [3]:

# Stage 5 - Build Logistic Regression pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(
        max_iter=300,
        class_weight="balanced",
        solver="saga",
        random_state=42
    ))
])

print("Logistic Regression pipeline created successfully!")


Logistic Regression pipeline created successfully!


In [2]:

# Restore Stage 5 data and feature lists

from google.colab import drive
import pandas as pd

drive.mount("/content/drive")

folder = "/content/drive/MyDrive/LendingClub_Project/stage4_data"

train_df = pd.read_parquet(
    folder + "/lending_club_train_stage4.parquet"
)

test_df = pd.read_parquet(
    folder + "/lending_club_test_stage4.parquet"
)

X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Training data:", X_train.shape)
print("Numerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))


Mounted at /content/drive
Training data: (1076280, 26)
Numerical features: 20
Categorical features: 6


In [4]:

# Stage 5 - Train Logistic Regression (Initial Experiment)

from sklearn.model_selection import train_test_split
import time

# Select a stratified sample from training data only
X_sample, _, y_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=100000,
    stratify=y_train,
    random_state=42
)

print("Training sample shape:", X_sample.shape)
print("Target distribution:")
print(y_sample.value_counts())

# Train the model
start_time = time.time()

logistic_model.fit(X_sample, y_sample)

training_time = time.time() - start_time

print("\nLogistic Regression trained successfully!")
print("Training time (seconds):", round(training_time, 2))


Training sample shape: (100000, 26)
Target distribution:
target
0    80035
1    19965
Name: count, dtype: int64

Logistic Regression trained successfully!
Training time (seconds): 48.75


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [5]:



from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.model_selection import train_test_split


X_test_sample, _, y_test_sample, _ = train_test_split(
    X_test,
    y_test,
    train_size=50000,
    stratify=y_test,
    random_state=42
)


y_pred = logistic_model.predict(X_test_sample)

print("Accuracy:", round(accuracy_score(y_test_sample, y_pred), 4))
print("Precision:", round(precision_score(y_test_sample, y_pred), 4))
print("Recall:", round(recall_score(y_test_sample, y_pred), 4))
print("F1-score:", round(f1_score(y_test_sample, y_pred), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_sample, y_pred))


Accuracy: 0.6465
Precision: 0.3184
Recall: 0.6752
F1-score: 0.4327

Confusion Matrix:
[[25586 14431]
 [ 3242  6741]]


In [6]:

# Stage 5 - Create training and validation sets

from sklearn.model_selection import train_test_split

X_model_train, X_val, y_model_train, y_val = train_test_split(
    X_sample,
    y_sample,
    test_size=0.20,
    stratify=y_sample,
    random_state=42
)

print("Model training:", X_model_train.shape)
print("Validation:", X_val.shape)

print("\nTraining target distribution:")
print(y_model_train.value_counts())

print("\nValidation target distribution:")
print(y_val.value_counts())


Model training: (80000, 26)
Validation: (20000, 26)

Training target distribution:
target
0    64028
1    15972
Name: count, dtype: int64

Validation target distribution:
target
0    16007
1     3993
Name: count, dtype: int64


In [7]:



import time

start_time = time.time()

logistic_model.fit(X_model_train, y_model_train)

training_time_lr = time.time() - start_time

print("Logistic Regression retrained successfully!")
print("Training time (seconds):", round(training_time_lr, 2))
print("Training records:", len(X_model_train))


Logistic Regression retrained successfully!
Training time (seconds): 32.94
Training records: 80000


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [8]:

# Stage 5 - Fix Logistic Regression convergence

from sklearn.linear_model import LogisticRegression
import time

# Update the classifier inside the existing pipeline
logistic_model.set_params(
    classifier=LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42
    )
)

start_time = time.time()

logistic_model.fit(X_model_train, y_model_train)

training_time_lr = time.time() - start_time

print("Logistic Regression retrained!")
print("Training time (seconds):", round(training_time_lr, 2))
print("Iterations used:", logistic_model.named_steps["classifier"].n_iter_)


Logistic Regression retrained!
Training time (seconds): 5.69
Iterations used: [115]


In [9]:

# Stage 5 - Logistic Regression Validation Results

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Predict on validation data
y_val_pred_lr = logistic_model.predict(X_val)
y_val_prob_lr = logistic_model.predict_proba(X_val)[:, 1]

print("Logistic Regression - Validation Results")
print("Accuracy:", round(accuracy_score(y_val, y_val_pred_lr), 4))
print("Precision:", round(precision_score(y_val, y_val_pred_lr), 4))
print("Recall:", round(recall_score(y_val, y_val_pred_lr), 4))
print("F1-score:", round(f1_score(y_val, y_val_pred_lr), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_val_prob_lr), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_lr))


Logistic Regression - Validation Results
Accuracy: 0.6462
Precision: 0.3176
Recall: 0.6719
F1-score: 0.4313
ROC-AUC: 0.7126

Confusion Matrix:
[[10242  5765]
 [ 1310  2683]]


In [10]:

# Stage 5 - Build Random Forest Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Random Forest does not require StandardScaler
preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest pipeline created successfully!")


Random Forest pipeline created successfully!


In [11]:



import time

start_time = time.time()

random_forest_model.fit(X_model_train, y_model_train)

training_time_rf = time.time() - start_time

print("Random Forest trained successfully!")
print("Training time (seconds):", round(training_time_rf, 2))
print("Training records:", len(X_model_train))


Random Forest trained successfully!
Training time (seconds): 51.11
Training records: 80000


In [12]:



from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


y_val_pred_rf = random_forest_model.predict(X_val)
y_val_prob_rf = random_forest_model.predict_proba(X_val)[:, 1]

print("Random Forest - Validation Results")
print("Accuracy:", round(accuracy_score(y_val, y_val_pred_rf), 4))
print("Precision:", round(precision_score(y_val, y_val_pred_rf), 4))
print("Recall:", round(recall_score(y_val, y_val_pred_rf), 4))
print("F1-score:", round(f1_score(y_val, y_val_pred_rf), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_val_prob_rf), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_rf))


Random Forest - Validation Results
Accuracy: 0.6936
Precision: 0.342
Recall: 0.5785
F1-score: 0.4298
ROC-AUC: 0.7129

Confusion Matrix:
[[11562  4445]
 [ 1683  2310]]


In [13]:

# Stage 5 - Compare validation results

import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_val, y_val_pred_lr),
        accuracy_score(y_val, y_val_pred_rf)
    ],
    "Precision": [
        precision_score(y_val, y_val_pred_lr),
        precision_score(y_val, y_val_pred_rf)
    ],
    "Recall": [
        recall_score(y_val, y_val_pred_lr),
        recall_score(y_val, y_val_pred_rf)
    ],
    "F1-score": [
        f1_score(y_val, y_val_pred_lr),
        f1_score(y_val, y_val_pred_rf)
    ],
    "ROC-AUC": [
        roc_auc_score(y_val, y_val_prob_lr),
        roc_auc_score(y_val, y_val_prob_rf)
    ]
})

display(comparison.round(4))


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.6462,0.3176,0.6719,0.4313,0.7126
1,Random Forest,0.6936,0.3420,0.5785,0.4298,0.7129
